# 06_step4_F2_applied_GCM_1deg_to_0.25deg - GCM 2025 no-hosing ERA5-like 1 deg -> 0.25 deg   [PLAN A]

The F2 downscaling (learned on ERA5, step3) is applied to the **GCM 2025 no-hosing adjusted by F1** ("ERA5-like 1 deg", notebook step2).

Result: `GCM_2025_nohosing_ERA5like_0.25deg`, the **target** of the RCM correction (notebook step5).

Europe only (`europe_bbox` from notebook 00).

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']
BBOX = {"lat": tuple(cfg["europe_bbox"]["lat"]), "lon": tuple(cfg["europe_bbox"]["lon"])}

RUN = "gcm_2025_nohosing"
USE_VALIDATION_F2 = False                # True: use F2 trained without 2015-2025 (tests)
BA1_DIR = bu.out_dir(cfg, "plan_A", "ba_1deg")
F2_DIR = bu.out_dir(cfg, "plan_A", "F2", "validation_without_2015_2025" if USE_VALIDATION_F2 else "full")
OUT_DIR = bu.out_dir(cfg, "plan_A", "era5like_025")
import f2_downscaler as f2

## 1. Inputs

In [ ]:
src = BA1_DIR / f"{RUN}_F1.nc"
assert src.exists(), f"{src.name} not found: run notebook step2."
sim_1deg = {v: bu.crop_bbox(bu.read_processed(src, v), BBOX) for v in VARS}

# target 0.25 deg grid = ERA5 0.25 deg grid over Europe (the one F2 was learned on)
era5_fine = bu.crop_bbox(bu.read_raw(cfg, "era5", "pr").isel(time=slice(0, 1)), BBOX)
tlat, tlon = era5_fine.lat.values, era5_fine.lon.values
print(f"GCM 1deg: {sim_1deg['pr'].sizes['lat']}x{sim_1deg['pr'].sizes['lon']} | target 0.25deg: {len(tlat)}x{len(tlon)}")

## 2. Application of F2 (supervisor's method: `f2_downscaler.apply_f2`)

In [ ]:
try:
    out = f2.apply_f2(sim_1deg, F2_DIR, tlat, tlon)
except NotImplementedError as e:
    out = None
    print("PLACEHOLDER:", e)

In [ ]:
if out is not None:
    dest = OUT_DIR / f"{RUN}_era5like_025.nc"
    bu.save_ds(xr.Dataset({v: out[v] for v in VARS}), dest)
    print("written:", dest)

## 3. Checks

In [ ]:
if out is not None:
    ds = xr.open_dataset(dest, chunks={"time": 365})
    s = ds.isel(time=slice(0, 365 * 3))
    print("grid identical to ERA5 0.25deg:", bool(bu.same_grid(ds["pr"], era5_fine)))
    print("pr min:", float(s["pr"].min()), "| frac tasmin>=tasmax (should be 0):", float(((s["tasmin"] >= s["tasmax"]) & s["tasmax"].notnull()).mean()),
          "| NaN pr:", float(s["pr"].isnull().mean()))
    # mean conservation: 1 deg (ERA5-like) vs 0.25 deg (same extent)
    for v in VARS:
        a = float(bu.area_mean(sim_1deg[v].isel(time=slice(0, 365))).mean().compute())
        b = float(bu.area_mean(ds[v].isel(time=slice(0, 365))).mean().compute())
        print(f"{v:7s} Europe mean 1deg = {a:.3f} | 0.25deg = {b:.3f}")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    m1 = sim_1deg["tasmax"].isel(time=slice(0, 90)).mean("time").compute()
    m2 = ds["tasmax"].isel(time=slice(0, 90)).mean("time").compute()
    for ax, m, t in zip(axes, (m1, m2), ("tasmax 1deg (ERA5-like)", "tasmax 0.25deg (after F2)")):
        im = ax.pcolormesh(m.lon, m.lat, m, shading="auto"); plt.colorbar(im, ax=ax, shrink=0.8); ax.set_title(t)
    plt.tight_layout(); plt.show()